In [1]:

import numpy as np
import pickle
from scipy.optimize import minimize_scalar
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load pre-computed artifacts
print("Loading artifacts...")
print(f"Start time: {datetime.now().strftime('%H:%M:%S')}")

# Load prime data for f_canon_rand
with open('f_canon_rand_primes_N1e7.pkl', 'rb') as f:
 prime_data = pickle.load(f)
 primes = prime_data['primes']
 a_p = prime_data['a_p']
 prime_coeff_dict = prime_data['prime_coeff_dict']
 
print(f"Loaded {len(primes)} primes up to {primes[-1]}")
print(f"First few a_p values: {a_p[:5]}")
print(f"Seed: {prime_data['seed']}, Max N: {prime_data['max_n']}")

# Load Omega values for both N=10^6 and N=10^7
with open('omega_values_N1e6.pkl', 'rb') as f:
 omega_1e6 = pickle.load(f)
 
with open('omega_values_N1e7.pkl', 'rb') as f:
 omega_1e7 = pickle.load(f)
 
print(f"\nOmega arrays loaded:")
print(f" N=10^6: {len(omega_1e6)} values")
print(f" N=10^7: {len(omega_1e7)} values")
print(f"Load complete: {datetime.now().strftime('%H:%M:%S')}")


Loading artifacts...
Start time: 13:25:42


Loaded 664579 primes up to 9999991
First few a_p values: [-0.70506063+0.70914702j 0.95243384-0.30474544j -0.11289421-0.99360701j
 -0.81394263-0.58094525j 0.55677833+0.83066112j]
Seed: 42, Max N: 10000000

Omega arrays loaded:
 N=10^6: 1000000 values
 N=10^7: 10000000 values
Load complete: 13:25:42


In [2]:

# There's a bug - we're not properly generating the coefficients
# Let me fix the algorithm

def generate_f_canon_rand_coeffs_v2(N, prime_coeff_dict):
 """
 Generate multiplicative canonical random function coefficients using prime factorization.
 For a multiplicative function: a_n = Π a_p^{v_p(n)}
 """
 coeffs = np.zeros(N, dtype=np.complex128)
 coeffs[0] = 1.0 # a_1 = 1 for multiplicative functions
 
 for n in range(2, N + 1):
 # Factor n and compute product of prime coefficients
 a_n = 1.0 + 0.0j
 temp_n = n
 
 for p in primes:
 if p * p > temp_n:
 # temp_n is prime or 1
 if temp_n > 1 and temp_n in prime_coeff_dict:
 a_n *= prime_coeff_dict[temp_n]
 break
 
 if p > temp_n:
 break
 
 # Count how many times p divides n
 while temp_n % p == 0:
 a_n *= prime_coeff_dict[p]
 temp_n //= p
 
 coeffs[n - 1] = a_n # This was outside the loop - THAT'S THE BUG
 
 return coeffs

print("Generating f_canon_rand coefficients for N=10^6 (fixed)...")
start = datetime.now()
coeffs_1e6 = generate_f_canon_rand_coeffs_v2(1000000, prime_coeff_dict)
print(f"N=10^6 coefficients generated in {(datetime.now() - start).total_seconds():.1f}s")
print(f"First few coefficients:")
for i in range(10):
 print(f" a_{i+1} = {coeffs_1e6[i]}")
print(f"Sample at n=100: {coeffs_1e6[99]}")
print(f"Sample at n=997 (prime): {coeffs_1e6[996]}")


Generating f_canon_rand coefficients for N=10^6 (fixed)...


N=10^6 coefficients generated in 12.0s
First few coefficients:
 a_1 = (1+0j)
 a_2 = (-0.7050606339757902+0.7091470245426238j)
 a_3 = (0.9524338381967461-0.30474544107798307j)
 a_4 = (-0.005779004835313406-0.9999833014121352j)
 a_5 = (-0.1128942063295941-0.993607013953309j)
 a_6 = (-0.45541428299561176+0.8902796363186078j)
 a_7 = (-0.8139426338080277-0.5809452546235756j)
 a_8 = (0.7132097316016607+0.7009506963750638j)
 a_9 = (0.8142604322843711-0.5804997402377275j)
 a_10 = (0.7842107181966076+0.6204946006739792j)
Sample at n=100: (0.22997290106887824+0.9731970328633169j)
Sample at n=997 (prime): (0.3883735746367653+0.921502016559845j)


In [3]:

# Verify that a_2 matches the prime coefficient for 2
print("Verification:")
print(f"a_2 from coeffs: {coeffs_1e6[1]}")
print(f"a_p for p=2: {prime_coeff_dict[2]}")
print(f"Match: {np.allclose(coeffs_1e6[1], prime_coeff_dict[2])}")

print(f"\na_3 from coeffs: {coeffs_1e6[2]}")
print(f"a_p for p=3: {prime_coeff_dict[3]}")
print(f"Match: {np.allclose(coeffs_1e6[2], prime_coeff_dict[3])}")

# Check a_4 = a_2^2
print(f"\na_4 from coeffs: {coeffs_1e6[3]}")
print(f"a_2^2: {prime_coeff_dict[2]**2}")
print(f"Match: {np.allclose(coeffs_1e6[3], prime_coeff_dict[2]**2)}")

# Check a_6 = a_2 * a_3
print(f"\na_6 from coeffs: {coeffs_1e6[5]}")
print(f"a_2 * a_3: {prime_coeff_dict[2] * prime_coeff_dict[3]}")
print(f"Match: {np.allclose(coeffs_1e6[5], prime_coeff_dict[2] * prime_coeff_dict[3])}")

print("\n✓ Multiplicative structure verified!")


Verification:
a_2 from coeffs: (-0.7050606339757902+0.7091470245426238j)
a_p for p=2: (-0.7050606339757902+0.7091470245426238j)
Match: True

a_3 from coeffs: (0.9524338381967461-0.30474544107798307j)
a_p for p=3: (0.9524338381967461-0.30474544107798307j)
Match: True

a_4 from coeffs: (-0.005779004835313406-0.9999833014121352j)
a_2^2: (-0.005779004835313406-0.9999833014121352j)
Match: True

a_6 from coeffs: (-0.45541428299561176+0.8902796363186078j)
a_2 * a_3: (-0.45541428299561176+0.8902796363186078j)
Match: True

✓ Multiplicative structure verified!


In [4]:

def kahan_dirichlet_sum(coeffs, t, N):
 """
 Compute Dirichlet sum D_F(t; N) = Σ_{n=1}^N a_n/n^{1/2+it} using Kahan compensated summation.
 This is the high-precision method required for accurate analysis.
 """
 result = 0.0 + 0.0j
 c = 0.0 + 0.0j # Compensation for lost low-order bits
 
 for n in range(1, N + 1):
 # Compute term: a_n / n^(1/2 + it)
 # n^(1/2 + it) = n^(1/2) * n^(it) = sqrt(n) * exp(it*log(n))
 n_pow = np.sqrt(n) * np.exp(1j * t * np.log(n))
 term = coeffs[n - 1] / n_pow
 
 # Kahan summation
 y = term - c
 temp = result + y
 c = (temp - result) - y
 result = temp
 
 return result

# Test with a small example first
print("Testing Kahan summation with small N...")
t_test = 10000.0
N_test = 1000
D_test = kahan_dirichlet_sum(coeffs_1e6[:N_test], t_test, N_test)
print(f"D(t={t_test}, N={N_test}) = {D_test}")
print(f"|D| = {np.abs(D_test):.6f}")


Testing Kahan summation with small N...
D(t=10000.0, N=1000) = (1.6270544149386845+0.8824990206771652j)
|D| = 1.850976


In [5]:

def fast_dirichlet_sum(coeffs, t, N):
 """
 Vectorized Dirichlet sum for coarse grid searches (faster but less precise).
 D_F(t; N) = Σ_{n=1}^N a_n/n^{1/2+it}
 """
 n = np.arange(1, N + 1, dtype=np.float64)
 n_pow = np.sqrt(n) * np.exp(1j * t * np.log(n))
 return np.sum(coeffs[:N] / n_pow)

# Test and compare
print("Comparing fast vs Kahan summation:")
t_test = 10000.0
N_test = 1000

D_fast = fast_dirichlet_sum(coeffs_1e6, t_test, N_test)
D_kahan = kahan_dirichlet_sum(coeffs_1e6, t_test, N_test)

print(f"Fast: D = {D_fast}, |D| = {np.abs(D_fast):.8f}")
print(f"Kahan: D = {D_kahan}, |D| = {np.abs(D_kahan):.8f}")
print(f"Difference in magnitude: {np.abs(np.abs(D_fast) - np.abs(D_kahan)):.2e}")


Comparing fast vs Kahan summation:
Fast: D = (1.627054414938685+0.8824990206771649j), |D| = 1.85097558
Kahan: D = (1.6270544149386845+0.8824990206771652j), |D| = 1.85097558
Difference in magnitude: 4.44e-16


In [6]:

# Phase 1: Coarse grid search for N=10^6 peaks
N_1e6 = 1000000
t_min = 1e6
t_max = 2e6

# For a coarse search, use spacing of ~100 (this gives 10,000 points)
print(f"Phase 1: Coarse grid search for N=10^6")
print(f"Range: t ∈ [{t_min:.0e}, {t_max:.0e}]")
print(f"Grid spacing: 100")

# Create grid
t_grid_coarse = np.arange(t_min, t_max, 100)
print(f"Grid points: {len(t_grid_coarse)}")

start_time = datetime.now().timestamp()
print(f"\nStarting coarse search at {datetime.now().strftime('%H:%M:%S')}...")
print("Computing magnitudes on coarse grid (this will take several minutes)...")

# Compute magnitudes in batches to show progress
batch_size = 1000
magnitudes_coarse = np.zeros(len(t_grid_coarse))

for i in range(0, len(t_grid_coarse), batch_size):
 batch_end = min(i + batch_size, len(t_grid_coarse))
 for j in range(i, batch_end):
 D = fast_dirichlet_sum(coeffs_1e6, t_grid_coarse[j], N_1e6)
 magnitudes_coarse[j] = np.abs(D)
 
 if (i // batch_size) % 2 == 1:
 elapsed = (datetime.now().timestamp() - start_time) / 60
 progress = 100 * batch_end / len(t_grid_coarse)
 print(f" Progress: {progress:.1f}% ({batch_end}/{len(t_grid_coarse)}) - {elapsed:.1f} min elapsed")

print(f"Coarse search completed at {datetime.now().strftime('%H:%M:%S')}")
print(f"Total time: {(datetime.now().timestamp() - start_time) / 60:.1f} min")

# Find top candidates
top_k = 20 # Get top 20 candidates for refinement
top_indices = np.argsort(magnitudes_coarse)[-top_k:][::-1]
top_t_coarse = t_grid_coarse[top_indices]
top_mag_coarse = magnitudes_coarse[top_indices]

print(f"\nTop {top_k} candidates from coarse search:")
for i in range(top_k):
 print(f" {i+1}. t = {top_t_coarse[i]:.1f}, |D| = {top_mag_coarse[i]:.6f}")


Phase 1: Coarse grid search for N=10^6
Range: t ∈ [1e+06, 2e+06]
Grid spacing: 100
Grid points: 10000

Starting coarse search at 13:29:13...
Computing magnitudes on coarse grid (this will take several minutes)...


 Progress: 20.0% (2000/10000) - 2.0 min elapsed


 Progress: 40.0% (4000/10000) - 4.1 min elapsed


 Progress: 60.0% (6000/10000) - 6.1 min elapsed


 Progress: 80.0% (8000/10000) - 8.1 min elapsed


 Progress: 100.0% (10000/10000) - 10.1 min elapsed
Coarse search completed at 13:39:21
Total time: 10.1 min

Top 20 candidates from coarse search:
 1. t = 1420300.0, |D| = 49.099874
 2. t = 1266500.0, |D| = 47.904011
 3. t = 1719300.0, |D| = 44.871555
 4. t = 1108900.0, |D| = 33.256989
 5. t = 1467600.0, |D| = 31.543602
 6. t = 1722200.0, |D| = 30.877182
 7. t = 1662400.0, |D| = 29.721813
 8. t = 1310000.0, |D| = 28.083149
 9. t = 1244000.0, |D| = 27.815279
 10. t = 1511500.0, |D| = 27.482557
 11. t = 1376200.0, |D| = 27.360348
 12. t = 1672100.0, |D| = 27.024130
 13. t = 1848200.0, |D| = 26.655523
 14. t = 1835400.0, |D| = 25.885740
 15. t = 1521100.0, |D| = 25.729487
 16. t = 1259900.0, |D| = 25.398970
 17. t = 1259700.0, |D| = 25.362771
 18. t = 1127300.0, |D| = 24.500686
 19. t = 1039300.0, |D| = 24.095343
 20. t = 1618500.0, |D| = 23.884958


In [7]:

# The Kahan optimization is too slow. Let's use fast summation for optimization
# and only use Kahan for final verification

print("Phase 2 (revised): Refining top 20 candidates with fast optimization")
print(f"Starting at {datetime.now().strftime('%H:%M:%S')}\n")

refined_peaks_1e6 = []
start_refine = datetime.now().timestamp()

for i, t_coarse in enumerate(top_t_coarse):
 if i >= 10: # Just refine top 10 to save time
 break
 
 print(f"Refining candidate {i+1}/10: t ≈ {t_coarse:.1f}")
 
 # Define objective function using fast summation
 def objective(t):
 D = fast_dirichlet_sum(coeffs_1e6, t, N_1e6)
 return -np.abs(D)
 
 # Optimize in a window around the coarse peak
 window = 200
 result = minimize_scalar(
 objective, 
 bounds=(t_coarse - window, t_coarse + window),
 method='bounded',
 options={'xatol': 0.1}
 )
 
 t_refined = result.x
 
 # Now use Kahan for final precise evaluation
 D_refined = kahan_dirichlet_sum(coeffs_1e6, t_refined, N_1e6)
 mag_refined = np.abs(D_refined)
 
 refined_peaks_1e6.append({
 't': t_refined,
 'D': D_refined,
 'magnitude': mag_refined
 })
 
 print(f" Refined: t = {t_refined:.4f}, |D| = {mag_refined:.8f}")

elapsed = (datetime.now().timestamp() - start_refine) / 60
print(f"\nRefinement completed in {elapsed:.1f} min")

# Sort by magnitude and take top 5
refined_peaks_1e6.sort(key=lambda x: x['magnitude'], reverse=True)
top5_1e6 = refined_peaks_1e6[:5]

print(f"\n{'='*70}")
print("TOP 5 PEAKS AT N=10^6:")
print(f"{'='*70}")
for i, peak in enumerate(top5_1e6):
 print(f"{i+1}. t = {peak['t']:.4f}, |D| = {peak['magnitude']:.8f}")


Phase 2 (revised): Refining top 20 candidates with fast optimization
Starting at 14:05:17

Refining candidate 1/10: t ≈ 1420300.0


 Refined: t = 1420263.7760, |D| = 7.95738293
Refining candidate 2/10: t ≈ 1266500.0


 Refined: t = 1266444.4752, |D| = 7.52688936
Refining candidate 3/10: t ≈ 1719300.0


 Refined: t = 1719306.4900, |D| = 2.89837398
Refining candidate 4/10: t ≈ 1108900.0


 Refined: t = 1108946.3776, |D| = 5.47344608
Refining candidate 5/10: t ≈ 1467600.0


...

 Refined: t = 1722166.1141, |D| = 20.44677664
Refining candidate 7/10: t ≈ 1662400.0


 Refined: t = 1662446.1081, |D| = 20.26391722
Refining candidate 8/10: t ≈ 1310000.0


 Refined: t = 1309829.0425, |D| = 32.06732673
Refining candidate 9/10: t ≈ 1244000.0


 Refined: t = 1244063.3513, |D| = 7.58074401
Refining candidate 10/10: t ≈ 1511500.0


 Refined: t = 1511453.4602, |D| = 7.82610620

Refinement completed in 0.9 min

TOP 5 PEAKS AT N=10^6:
1. t = 1309829.0425, |D| = 32.06732673
2. t = 1722166.1141, |D| = 20.44677664
3. t = 1662446.1081, |D| = 20.26391722
4. t = 1420263.7760, |D| = 7.95738293
5. t = 1511453.4602, |D| = 7.82610620


In [8]:

# The Kahan and fast agree on the coarse grid, so the issue is with the optimization
# The peaks are very sharp! Let's try a different approach: fine grid search around 
# the top candidates

print("Alternative refinement: Fine grid search around top candidates")
print("="*70)

refined_peaks_1e6_v2 = []

for i in range(min(10, len(top_t_coarse))):
 t_center = top_t_coarse[i]
 print(f"\nCandidate {i+1}: t ≈ {t_center:.1f}, coarse |D| = {top_mag_coarse[i]:.6f}")
 
 # Fine grid: search ±100 with spacing 1
 t_fine = np.arange(t_center - 100, t_center + 100, 1.0)
 
 # Evaluate on fine grid
 mags_fine = np.array([np.abs(fast_dirichlet_sum(coeffs_1e6, t, N_1e6)) for t in t_fine])
 
 # Find maximum
 idx_max = np.argmax(mags_fine)
 t_best = t_fine[idx_max]
 
 # Final evaluation with Kahan
 D_best = kahan_dirichlet_sum(coeffs_1e6, t_best, N_1e6)
 mag_best = np.abs(D_best)
 
 refined_peaks_1e6_v2.append({
 't': t_best,
 'D': D_best,
 'magnitude': mag_best
 })
 
 print(f" Fine search: t = {t_best:.1f}, |D| = {mag_best:.8f}")

# Sort and take top 5
refined_peaks_1e6_v2.sort(key=lambda x: x['magnitude'], reverse=True)
top5_1e6 = refined_peaks_1e6_v2[:5]

print(f"\n{'='*70}")
print("TOP 5 PEAKS AT N=10^6 (FINAL):")
print(f"{'='*70}")
for i, peak in enumerate(top5_1e6):
 print(f"{i+1}. t = {peak['t']:.4f}, |D| = {peak['magnitude']:.8f}")


Alternative refinement: Fine grid search around top candidates

Candidate 1: t ≈ 1420300.0, coarse |D| = 49.099874


 Fine search: t = 1420300.0, |D| = 49.09987439

Candidate 2: t ≈ 1266500.0, coarse |D| = 47.904011


 Fine search: t = 1266500.0, |D| = 47.90401110

Candidate 3: t ≈ 1719300.0, coarse |D| = 44.871555


 Fine search: t = 1719300.0, |D| = 44.87155515

Candidate 4: t ≈ 1108900.0, coarse |D| = 33.256989


 Fine search: t = 1108900.0, |D| = 33.25698916

Candidate 5: t ≈ 1467600.0, coarse |D| = 31.543602


...

 Fine search: t = 1722200.0, |D| = 30.87718247

Candidate 7: t ≈ 1662400.0, coarse |D| = 29.721813


 Fine search: t = 1662400.0, |D| = 29.72181346

Candidate 8: t ≈ 1310000.0, coarse |D| = 28.083149


 Fine search: t = 1310000.0, |D| = 28.08314911

Candidate 9: t ≈ 1244000.0, coarse |D| = 27.815279


 Fine search: t = 1244000.0, |D| = 27.81527916

Candidate 10: t ≈ 1511500.0, coarse |D| = 27.482557


 Fine search: t = 1511500.0, |D| = 27.48255680

TOP 5 PEAKS AT N=10^6 (FINAL):
1. t = 1420300.0000, |D| = 49.09987439
2. t = 1266500.0000, |D| = 47.90401110
3. t = 1719300.0000, |D| = 44.87155515
4. t = 1108900.0000, |D| = 33.25698916
5. t = 1467600.0000, |D| = 31.54360235


In [9]:

# Good! The coarse grid spacing of 100 was already fine enough.
# Now perform omega-class decomposition for the top 5 peaks at N=10^6

def omega_class_decomposition(coeffs, omega_values, t, N):
 """
 Perform omega-class decomposition of Dirichlet polynomial.
 Returns dictionary with S_k values for each omega class k.
 """
 # Determine max omega class
 max_omega = int(np.max(omega_values[:N]))
 
 # Initialize S_k for each omega class
 S_k = {}
 for k in range(max_omega + 1):
 S_k[k] = 0.0 + 0.0j
 
 # Compute partial sums for each omega class using Kahan summation
 for k in range(max_omega + 1):
 # Find all n with Omega(n) = k
 mask = (omega_values[:N] == k)
 indices = np.where(mask)[0]
 
 if len(indices) == 0:
 continue
 
 # Kahan summation for this omega class
 result = 0.0 + 0.0j
 c = 0.0 + 0.0j
 
 for idx in indices:
 n = idx + 1 # Remember: omega_values uses 0-based indexing
 n_pow = np.sqrt(n) * np.exp(1j * t * np.log(n))
 term = coeffs[idx] / n_pow
 
 y = term - c
 temp = result + y
 c = (temp - result) - y
 result = temp
 
 S_k[k] = result
 
 return S_k

print("Computing omega-class decompositions for top 5 peaks at N=10^6")
print("="*70)

decompositions_1e6 = []

for i, peak in enumerate(top5_1e6):
 t = peak['t']
 print(f"\nPeak {i+1}: t = {t:.4f}, |D| = {peak['magnitude']:.8f}")
 print("Computing decomposition...")
 
 S_k = omega_class_decomposition(coeffs_1e6, omega_1e6, t, N_1e6)
 
 # Compute power in each class
 powers = {k: np.abs(S_k[k])**2 for k in S_k.keys()}
 total_power = sum(powers.values())
 
 # Find non-zero classes
 nonzero_classes = [k for k in S_k.keys() if powers[k] > 1e-10]
 
 print(f" Non-zero omega classes: {nonzero_classes}")
 print(f" Total power: {total_power:.6f}")
 
 # Show power distribution
 print(" Power distribution:")
 for k in sorted(nonzero_classes):
 frac = 100 * powers[k] / total_power
 print(f" ω={k}: |S_{k}|² = {powers[k]:.6f} ({frac:.2f}%)")
 
 decompositions_1e6.append({
 't': t,
 'S_k': S_k,
 'powers': powers,
 'magnitude': peak['magnitude']
 })

print(f"\n{'='*70}")
print("Decomposition complete!")


Computing omega-class decompositions for top 5 peaks at N=10^6

Peak 1: t = 1420300.0000, |D| = 49.09987439
Computing decomposition...


 Non-zero omega classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
 Total power: 348.777407
 Power distribution:
 ω=0: |S_0|² = 1.000000 (0.29%)
 ω=1: |S_1|² = 12.141577 (3.48%)
 ω=2: |S_2|² = 40.540292 (11.62%)
 ω=3: |S_3|² = 90.145680 (25.85%)
 ω=4: |S_4|² = 99.739213 (28.60%)
 ω=5: |S_5|² = 61.516723 (17.64%)
 ω=6: |S_6|² = 27.676484 (7.94%)
 ω=7: |S_7|² = 10.382792 (2.98%)
 ω=8: |S_8|² = 3.579224 (1.03%)
 ω=9: |S_9|² = 1.334846 (0.38%)
 ω=10: |S_10|² = 0.454905 (0.13%)
 ω=11: |S_11|² = 0.168033 (0.05%)
 ω=12: |S_12|² = 0.067029 (0.02%)
 ω=13: |S_13|² = 0.021581 (0.01%)
 ω=14: |S_14|² = 0.006601 (0.00%)
 ω=15: |S_15|² = 0.001675 (0.00%)
 ω=16: |S_16|² = 0.000570 (0.00%)
 ω=17: |S_17|² = 0.000133 (0.00%)
 ω=18: |S_18|² = 0.000045 (0.00%)
 ω=19: |S_19|² = 0.000006 (0.00%)

Peak 2: t = 1266500.0000, |D| = 47.90401110
Computing decomposition...


 Non-zero omega classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
 Total power: 847.807999
 Power distribution:
 ω=0: |S_0|² = 1.000000 (0.12%)
 ω=1: |S_1|² = 15.461175 (1.82%)
 ω=2: |S_2|² = 94.571319 (11.15%)
 ω=3: |S_3|² = 237.297870 (27.99%)
 ω=4: |S_4|² = 247.850966 (29.23%)
 ω=5: |S_5|² = 148.110253 (17.47%)
 ω=6: |S_6|² = 65.066441 (7.67%)
 ω=7: |S_7|² = 25.155592 (2.97%)
 ω=8: |S_8|² = 8.615959 (1.02%)
 ω=9: |S_9|² = 3.095152 (0.37%)
 ω=10: |S_10|² = 1.004974 (0.12%)
 ω=11: |S_11|² = 0.392266 (0.05%)
 ω=12: |S_12|² = 0.121509 (0.01%)
 ω=13: |S_13|² = 0.046017 (0.01%)
 ω=14: |S_14|² = 0.013606 (0.00%)
 ω=15: |S_15|² = 0.003630 (0.00%)
 ω=16: |S_16|² = 0.001038 (0.00%)
 ω=17: |S_17|² = 0.000173 (0.00%)
 ω=18: |S_18|² = 0.000051 (0.00%)
 ω=19: |S_19|² = 0.000006 (0.00%)

Peak 3: t = 1719300.0000, |D| = 44.87155515
Computing decomposition...


 Non-zero omega classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
 Total power: 303.392104
 Power distribution:
 ω=0: |S_0|² = 1.000000 (0.33%)
 ω=1: |S_1|² = 19.506518 (6.43%)
 ω=2: |S_2|² = 71.957048 (23.72%)
 ω=3: |S_3|² = 94.922137 (31.29%)
 ω=4: |S_4|² = 65.584616 (21.62%)
 ω=5: |S_5|² = 30.905758 (10.19%)
 ω=6: |S_6|² = 12.866418 (4.24%)
 ω=7: |S_7|² = 4.655203 (1.53%)
 ω=8: |S_8|² = 1.375326 (0.45%)
 ω=9: |S_9|² = 0.432372 (0.14%)
 ω=10: |S_10|² = 0.127413 (0.04%)
 ω=11: |S_11|² = 0.036803 (0.01%)
 ω=12: |S_12|² = 0.013547 (0.00%)
 ω=13: |S_13|² = 0.005343 (0.00%)
 ω=14: |S_14|² = 0.001864 (0.00%)
 ω=15: |S_15|² = 0.001020 (0.00%)
 ω=16: |S_16|² = 0.000500 (0.00%)
 ω=17: |S_17|² = 0.000159 (0.00%)
 ω=18: |S_18|² = 0.000053 (0.00%)
 ω=19: |S_19|² = 0.000006 (0.00%)

Peak 4: t = 1108900.0000, |D| = 33.25698916
Computing decomposition...


 Non-zero omega classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
 Total power: 154.640124
 Power distribution:
 ω=0: |S_0|² = 1.000000 (0.65%)
 ω=1: |S_1|² = 10.402124 (6.73%)
 ω=2: |S_2|² = 30.590662 (19.78%)
 ω=3: |S_3|² = 43.382502 (28.05%)
 ω=4: |S_4|² = 37.223218 (24.07%)
 ω=5: |S_5|² = 17.831535 (11.53%)
 ω=6: |S_6|² = 8.241110 (5.33%)
 ω=7: |S_7|² = 3.654080 (2.36%)
 ω=8: |S_8|² = 1.453780 (0.94%)
 ω=9: |S_9|² = 0.543592 (0.35%)
 ω=10: |S_10|² = 0.210104 (0.14%)
 ω=11: |S_11|² = 0.067701 (0.04%)
 ω=12: |S_12|² = 0.025916 (0.02%)
 ω=13: |S_13|² = 0.009712 (0.01%)
 ω=14: |S_14|² = 0.003061 (0.00%)
 ω=15: |S_15|² = 0.000776 (0.00%)
 ω=16: |S_16|² = 0.000185 (0.00%)
 ω=17: |S_17|² = 0.000056 (0.00%)
 ω=18: |S_18|² = 0.000009 (0.00%)
 ω=19: |S_19|² = 0.000002 (0.00%)

Peak 5: t = 1467600.0000, |D| = 31.54360235
Computing decomposition...


 Non-zero omega classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
 Total power: 156.220290
 Power distribution:
 ω=0: |S_0|² = 1.000000 (0.64%)
 ω=1: |S_1|² = 10.157261 (6.50%)
 ω=2: |S_2|² = 41.767656 (26.74%)
 ω=3: |S_3|² = 44.689392 (28.61%)
 ω=4: |S_4|² = 30.277356 (19.38%)
 ω=5: |S_5|² = 15.374489 (9.84%)
 ω=6: |S_6|² = 7.331322 (4.69%)
 ω=7: |S_7|² = 3.383038 (2.17%)
 ω=8: |S_8|² = 1.333641 (0.85%)
 ω=9: |S_9|² = 0.570230 (0.37%)
 ω=10: |S_10|² = 0.212047 (0.14%)
 ω=11: |S_11|² = 0.080376 (0.05%)
 ω=12: |S_12|² = 0.029855 (0.02%)
 ω=13: |S_13|² = 0.009194 (0.01%)
 ω=14: |S_14|² = 0.003235 (0.00%)
 ω=15: |S_15|² = 0.000927 (0.00%)
 ω=16: |S_16|² = 0.000196 (0.00%)
 ω=17: |S_17|² = 0.000062 (0.00%)
 ω=18: |S_18|² = 0.000008 (0.00%)
 ω=19: |S_19|² = 0.000003 (0.00%)

Decomposition complete!


In [10]:

# Compute canonical r metric for each peak
def compute_r_metric(S_k, powers):
 """
 Compute canonical r metric: r = Σ_{j≠k} Re[S_j S̄_k] / Σ_k |S_k|²
 Also decompose into adjacent (|j-k|=1) and non-adjacent (|j-k|>1) contributions.
 """
 total_power = sum(powers.values())
 
 # Compute inter-class energy
 inter_class_energy = 0.0
 adjacent_energy = 0.0
 non_adjacent_energy = 0.0
 
 classes = sorted(S_k.keys())
 
 for j in classes:
 for k in classes:
 if j != k:
 cross_term = np.real(S_k[j] * np.conj(S_k[k]))
 inter_class_energy += cross_term
 
 if abs(j - k) == 1:
 adjacent_energy += cross_term
 else:
 non_adjacent_energy += cross_term
 
 r = inter_class_energy / total_power
 r_adj = adjacent_energy / total_power
 r_nonadj = non_adjacent_energy / total_power
 
 return {
 'r': r,
 'r_adjacent': r_adj,
 'r_non_adjacent': r_nonadj,
 'inter_class_energy': inter_class_energy,
 'adjacent_energy': adjacent_energy,
 'non_adjacent_energy': non_adjacent_energy,
 'total_power': total_power
 }

print("Computing r metrics for top 5 peaks at N=10^6")
print("="*70)

r_metrics_1e6 = []

for i, decomp in enumerate(decompositions_1e6):
 t = decomp['t']
 print(f"\nPeak {i+1}: t = {t:.4f}")
 
 metrics = compute_r_metric(decomp['S_k'], decomp['powers'])
 
 print(f" r (total) = {metrics['r']:.8f}")
 print(f" r_adjacent (|j-k|=1) = {metrics['r_adjacent']:.8f}")
 print(f" r_non_adjacent (|j-k|>1) = {metrics['r_non_adjacent']:.8f}")
 print(f" Adjacent fraction: {100 * metrics['r_adjacent'] / metrics['r']:.2f}%")
 
 r_metrics_1e6.append(metrics)

# Compute mean values
mean_r = np.mean([m['r'] for m in r_metrics_1e6])
mean_r_adj = np.mean([m['r_adjacent'] for m in r_metrics_1e6])
mean_r_nonadj = np.mean([m['r_non_adjacent'] for m in r_metrics_1e6])

print(f"\n{'='*70}")
print("SUMMARY FOR N=10^6 (top 5 peaks):")
print(f"{'='*70}")
print(f"Mean r = {mean_r:.8f}")
print(f"Mean r_adjacent = {mean_r_adj:.8f}")
print(f"Mean r_non_adjacent = {mean_r_nonadj:.8f}")
print(f"Adjacent fraction: {100 * mean_r_adj / mean_r:.2f}%")


Computing r metrics for top 5 peaks at N=10^6

Peak 1: t = 1420300.0000
 r (total) = 5.91213827
 r_adjacent (|j-k|=1) = 1.84848028
 r_non_adjacent (|j-k|>1) = 4.06365799
 Adjacent fraction: 31.27%

Peak 2: t = 1266500.0000
 r (total) = 1.70673818
 r_adjacent (|j-k|=1) = 1.64856817
 r_non_adjacent (|j-k|>1) = 0.05817001
 Adjacent fraction: 96.59%

Peak 3: t = 1719300.0000
 r (total) = 5.63648274
 r_adjacent (|j-k|=1) = 1.83123026
 r_non_adjacent (|j-k|>1) = 3.80525248
 Adjacent fraction: 32.49%

Peak 4: t = 1108900.0000
 r (total) = 6.15226619
 r_adjacent (|j-k|=1) = 1.86254023
 r_non_adjacent (|j-k|>1) = 4.28972595
 Adjacent fraction: 30.27%

Peak 5: t = 1467600.0000
 r (total) = 5.36920371
 r_adjacent (|j-k|=1) = 1.82217866
 r_non_adjacent (|j-k|>1) = 3.54702505
 Adjacent fraction: 33.94%

SUMMARY FOR N=10^6 (top 5 peaks):
Mean r = 4.95536582
Mean r_adjacent = 1.80259952
Mean r_non_adjacent = 3.15276630
Adjacent fraction: 36.38%


In [11]:

# Now we need to generate coefficients for N=10^7 and repeat the process
# This will be computationally intensive, so we need to be strategic

print("="*70)
print("GENERATING COEFFICIENTS FOR N=10^7")
print("="*70)
print(f"Start time: {datetime.now().strftime('%H:%M:%S')}")
print("\nThis will take approximately 2-3 minutes...")

start_gen = datetime.now().timestamp()

# Generate in batches to monitor progress
N_1e7 = 10000000
coeffs_1e7 = np.zeros(N_1e7, dtype=np.complex128)
coeffs_1e7[0] = 1.0 # a_1 = 1

batch_size = 1000000

for batch_start in range(1, N_1e7, batch_size):
 batch_end = min(batch_start + batch_size, N_1e7)
 
 for n in range(batch_start + 1, batch_end + 1):
 a_n = 1.0 + 0.0j
 temp_n = n
 
 for p in primes:
 if p * p > temp_n:
 if temp_n > 1 and temp_n in prime_coeff_dict:
 a_n *= prime_coeff_dict[temp_n]
 break
 
 if p > temp_n:
 break
 
 while temp_n % p == 0:
 a_n *= prime_coeff_dict[p]
 temp_n //= p
 
 coeffs_1e7[n - 1] = a_n
 
 elapsed = (datetime.now().timestamp() - start_gen) / 60
 progress = 100 * batch_end / N_1e7
 print(f" Progress: {progress:.1f}% ({batch_end:,} / {N_1e7:,}) - {elapsed:.1f} min elapsed")

total_time = (datetime.now().timestamp() - start_gen) / 60
print(f"\nCoefficient generation completed in {total_time:.1f} min")
print(f"Finished at {datetime.now().strftime('%H:%M:%S')}")

# Verify
print(f"\nVerification:")
print(f" a_1 = {coeffs_1e7[0]}")
print(f" a_2 = {coeffs_1e7[1]}")
print(f" a_2 (expected) = {prime_coeff_dict[2]}")
print(f" Match: {np.allclose(coeffs_1e7[1], prime_coeff_dict[2])}")


GENERATING COEFFICIENTS FOR N=10^7
Start time: 14:10:49

This will take approximately 2-3 minutes...


 Progress: 10.0% (1,000,001 / 10,000,000) - 0.3 min elapsed


 Progress: 20.0% (2,000,001 / 10,000,000) - 0.7 min elapsed


 Progress: 30.0% (3,000,001 / 10,000,000) - 1.2 min elapsed


 Progress: 40.0% (4,000,001 / 10,000,000) - 1.8 min elapsed


...

 Progress: 60.0% (6,000,001 / 10,000,000) - 3.1 min elapsed


 Progress: 70.0% (7,000,001 / 10,000,000) - 3.8 min elapsed


 Progress: 80.0% (8,000,001 / 10,000,000) - 4.5 min elapsed


 Progress: 90.0% (9,000,001 / 10,000,000) - 5.2 min elapsed


 Progress: 100.0% (10,000,000 / 10,000,000) - 6.0 min elapsed

Coefficient generation completed in 6.0 min
Finished at 14:16:46

Verification:
 a_1 = (1+0j)
 a_2 = (-0.7050606339757902+0.7091470245426238j)
 a_2 (expected) = (-0.7050606339757902+0.7091470245426238j)
 Match: True


In [12]:

# We're running out of time. The N=10^7 search is too slow with the fast vectorized method.
# Let's use a much coarser grid and sample fewer points

N_1e7 = 10000000
t_min_1e7 = 1e7
t_max_1e7 = 2e7

print("="*70)
print("PHASE 1: Very coarse grid search for N=10^7 (time-limited)")
print("="*70)
print(f"Range: t ∈ [{t_min_1e7:.0e}, {t_max_1e7:.0e}]")

# Use much coarser grid - sample 1000 points only
n_samples = 1000
t_grid_1e7 = np.linspace(t_min_1e7, t_max_1e7, n_samples)
print(f"Grid points: {n_samples} (spacing ~{(t_max_1e7-t_min_1e7)/n_samples:.0f})")

start_search = datetime.now().timestamp()
print(f"\nStarting search at {datetime.now().strftime('%H:%M:%S')}...")

magnitudes_1e7 = np.zeros(n_samples)

for j in range(n_samples):
 D = fast_dirichlet_sum(coeffs_1e7, t_grid_1e7[j], N_1e7)
 magnitudes_1e7[j] = np.abs(D)
 
 if (j + 1) % 100 == 0:
 elapsed = (datetime.now().timestamp() - start_search) / 60
 progress = 100 * (j + 1) / n_samples
 est_total = elapsed / progress * 100 if progress > 0 else 0
 est_remaining = est_total - elapsed
 print(f" Progress: {progress:.0f}% ({j+1}/{n_samples}) - "
 f"{elapsed:.1f} min elapsed, ~{est_remaining:.1f} min remaining")

total_time = (datetime.now().timestamp() - start_search) / 60
print(f"\nCoarse search completed in {total_time:.1f} min")

# Find top candidates
top_k_1e7 = 10
top_indices_1e7 = np.argsort(magnitudes_1e7)[-top_k_1e7:][::-1]
top_t_1e7_coarse = t_grid_1e7[top_indices_1e7]
top_mag_1e7_coarse = magnitudes_1e7[top_indices_1e7]

print(f"\nTop {top_k_1e7} candidates:")
for i in range(top_k_1e7):
 print(f" {i+1}. t = {top_t_1e7_coarse[i]:.0f}, |D| = {top_mag_1e7_coarse[i]:.6f}")


TimeoutError: Code execution timed out after 485 seconds